## 1. PELT

### 1.0 Import libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
import re
import statsmodels.api as sm
import pyhomogeneity as ph
import math
from IPython.display import display
import ruptures as rpt
from typing import List
from math import log
from typing import List, Dict, Any
import seaborn as sns
from pandas.plotting import register_matplotlib_converters
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose

### 1.1 Pre-processing functions

In [ ]:
pm10_path = r"C:\MyPython\PM-Trends\Trends_PM10"

def limpiar_nombres_columnas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def _pick_pm10_col(df: pd.DataFrame) -> str:
    # intenta 'pm10' exacto, si no busca por variantes
    if 'pm10' in df.columns:
        return 'pm10'
    cand = [c for c in df.columns if re.fullmatch(r'(?i)\s*pm\s*10\s*', str(c))]
    if cand:
        return cand[0]
    # fallback: cualquier columna que contenga 'pm10'
    cand = [c for c in df.columns if 'pm10' in str(c).lower()]
    if cand:
        return cand[0]
    raise KeyError("No encontré columna de pm10")

def _ensure_datetime_index(df: pd.DataFrame, fecha_col='FECHA') -> pd.DataFrame:
    df = df.copy()
    if fecha_col not in df.columns:
        # intenta variantes comunes
        posibles = [c for c in df.columns if 'FECHA' in str(c).lower() or 'date' in str(c).lower()]
        if not posibles:
            raise KeyError("No encontré columna FECHA")
        fecha_col = posibles[0]
    df[fecha_col] = pd.to_datetime(df[fecha_col], errors='coerce')
    df = df.dropna(subset=[fecha_col]).sort_values(fecha_col).set_index(fecha_col)
    # fuerza frecuencia diaria y rellena huecos con interpolación temporal
    df = df.asfreq('D')
    return df

def _site_name_from_filename(path: str) -> str:
    base = os.path.splitext(os.path.basename(path))[0]
    # Quita sufijos típicos como "_IMPUTADOS_DIAS"
    base = re.sub(r'_IMPUTADOS?_DIAS$', '', base, flags=re.IGNORECASE)
    return base

def _save_plot(fig, out_path: str):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, bbox_inches='tight', dpi=150)
    plt.close(fig)

def procesar_pm10_folder(
    folder: str,
    out_dir_plots: str = None,
    out_dir_csv: str = None,
    stl_monthly_period: int = 12,
    loess_frac: float = 0.4
):
    """
    Recorre todos los .xlsx en 'folder' y genera:
        - STL mensual (period=12)
        - Tendencia anual por LOESS
    Guarda PNGs y CSVs (si se indican out_dir_*).
    Devuelve un resumen por sitio con shapes de cada serie.
    """
    if out_dir_plots is None:
        out_dir_plots = os.path.join(folder, "_plots")
    if out_dir_csv is None:
        out_dir_csv  = os.path.join(folder, "_series")

    os.makedirs(out_dir_plots, exist_ok=True)
    os.makedirs(out_dir_csv, exist_ok=True)

    resumen = []
    series_ajustadas = {}
    xlsx_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".xlsx")]
    for fpath in xlsx_files:
        try:
            sitio = _site_name_from_filename(fpath)
            df = pd.read_excel(fpath)
            df = limpiar_nombres_columnas(df)

            pm_col = _pick_pm10_col(df)
            ser = df[[pm_col, 'FECHA']] if 'FECHA' in df.columns else df[[pm_col]]
            ser = ser.rename(columns={pm_col: 'pm10'})
            ser = _ensure_datetime_index(ser, 'FECHA')
            #ser['pm10'] = _safe_interpolate(ser['pm10'])
            ser = ser.dropna(subset=['pm10'])

            # ---------- STL MENSUAL ----------
            monthly = ser.resample('ME').mean()
            monthly = monthly.dropna()
            if len(monthly) >= stl_monthly_period * 2:
                stl_m = STL(monthly['pm10'], period=stl_monthly_period, robust=True).fit()
                fig_m = stl_m.plot()
                fig_m.suptitle(f"STL Mensual (period=12) - {sitio}", y=1.02)
                _save_plot(fig_m, os.path.join(out_dir_plots, f"{sitio}_stl_mensual.png"))

                # Calculate amplitude of seasonal component over time
                seasonal = stl_m.seasonal
                # Calculate amplitude (peak-to-trough difference) over time
                amplitude = seasonal.resample('YE').apply(lambda x: x.max() - x.min())
                # Plot the seasonal component and amplitude
                fig, ax = plt.subplots(2, 1, figsize=(12, 8))
                # Plot seasonal component
                ax[0].plot(seasonal)
                ax[0].set_title('Seasonal Component')
                # Plot amplitude
                ax[1].plot(amplitude)
                ax[1].set_title('Amplitude of Seasonal Component')
                _save_plot(fig, os.path.join(out_dir_plots, f"{sitio}_stl_amplitude.png"))
                #plt.tight_layout()
                #plt.show()

                # Print amplitude values
                print(f"Amplitude values for {sitio}:")
                print(amplitude)

                # Fit a linear trend to the trend component
                trend = stl_m.trend.dropna()
                X = np.arange(len(trend)).reshape(-1, 1)
                y = trend.values
                model = sm.OLS(y, sm.add_constant(X)).fit()
                ci = model.conf_int(alpha=0.05)
                slope_ci = ci[1]
                slope = model.params[1]
                print(f"STL Mensual - {sitio}")
                print(f"Slope: {slope:.6f}; 95% CI: [{slope_ci[0]:.6f}, {slope_ci[1]:.6f}]")
                print(model.summary())

                comp_m = pd.DataFrame({
                    'observed': monthly['pm10'],
                    'trend': stl_m.trend,
                    'seasonal': stl_m.seasonal,
                    'resid': stl_m.resid
                })
                
                # ---------- SERIE AJUSTADA ESTACIONALMENTE (SA) ----------
                # SA = Trend + Remainder = Observed - Seasonal(t)
                # Usamos el componente estacional de STL (no una climatología fija de 12 meses),
                # porque la Fig. 2 muestra que la amplitud estacional cambia con el tiempo:
                # una climatología fija dejaría "fugas" estacionales residuales en la serie ajustada.
                comp_m['sa'] = comp_m['trend'] + comp_m['resid']

                # Chequeo de consistencia (debe ser ~0 en el esquema aditivo de STL)
                recon_err = (comp_m['observed'] - (comp_m['trend'] + comp_m['seasonal'] + comp_m['resid'])).abs().max()
                if recon_err > 1e-6:
                    print(f"[WARN] {sitio}: descomposición STL no aditiva o con NaNs residuales (max err={recon_err:.4f})")

                # Guarda la serie SA por separado (para uso directo aguas abajo en el CPD)
                sa_out = comp_m[['observed', 'sa']].rename(columns={'sa': 'pm10_sa'})
                sa_out.to_csv(os.path.join(out_dir_csv, f"{sitio}_stl_sa_mensual.csv"), encoding='utf-8-sig')

                # Gráfico comparativo: observado vs SA
                fig_sa, ax_sa = plt.subplots(figsize=(12, 5))
                ax_sa.plot(comp_m.index, comp_m['observed'], label='PM10 mensual (observado)', alpha=0.5)
                ax_sa.plot(comp_m.index, comp_m['sa'], label='PM10 mensual (ajustado estacionalmente, STL)', linewidth=2)
                ax_sa.set_title(f"Serie ajustada estacionalmente (STL) - {sitio}")
                ax_sa.set_xlabel("Fecha"); ax_sa.set_ylabel("PM10 (µg/m³)")
                ax_sa.legend()
                _save_plot(fig_sa, os.path.join(out_dir_plots, f"{sitio}_stl_sa_mensual.png"))

                # Acumula en el diccionario que se retornará para uso directo en el CPD
                series_ajustadas[sitio] = comp_m['sa'].rename(sitio)
                
                comp_m.to_csv(os.path.join(out_dir_csv, f"{sitio}_stl_mensual.csv"), encoding='utf-8-sig')
            else:
                comp_m = pd.DataFrame()

            resumen.append({
                'sitio': sitio,
                'meses': int(len(monthly))
            })
        except Exception as e:
            resumen.append({'sitio': os.path.basename(fpath), 'error': str(e)})

    return pd.DataFrame(resumen), series_ajustadas

# --- Ejemplo de uso ---
df_resumen, series_ajustadas = procesar_pm10_folder(pm10_path)
df_resumen


### 1.2 Hyperparameter search

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ruptures as rpt
from math import log
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata

# ---------------------------------------------------------------------------
# Read the series and prepare the signal for change point detection
# ---------------------------------------------------------------------------
site_path = r"C:\MyPython\PM-Trends\Trends_PM10\pm10_mensual_sa\SW_IMPUTADOS_stl_sa_mensual.csv"
outfile   = r"C:\MyPython\PM-Trends\Trends_PM10\PELT results cLinear_sa\ruptures_hyperp_tuning_SW_cLinear.csv"

df_site = pd.read_csv(site_path)
df_site = limpiar_nombres_columnas(df_site)

pm_col = _pick_pm10_col(df_site)
ser = df_site[[pm_col, 'FECHA']] if 'FECHA' in df_site.columns else df_site[[pm_col]]
ser = ser.rename(columns={pm_col: 'pm10_sa'})
ser = _ensure_datetime_index(ser, 'FECHA')
ser = ser.dropna(subset=['pm10_sa'])

# Monthly series to work with
mensual = ser.dropna()
mensual = mensual.sort_index()

signal = mensual.values.reshape(-1, 1)
n = len(signal)
print(f"Loaded series with {n} points for change point detection.")

# =====================================================================
# 1) Variance-based penalty bounds
#    NOTE: `signal` here is the SEASONALLY-ADJUSTED series
#    (SA = Trend + Remainder = Observed - Seasonal(t)), NOT raw data.
#    Because seasonal swings are already removed, the first-difference
#    variance mostly reflects Remainder noise + residual Trend curvature
#    -- it will be smaller than if computed on the raw observed series,
#    and that is expected.
# =====================================================================
n = len(signal)

# ---- Robust noise variance via first differences of the SA series ----
diffs_sa = np.diff(signal.ravel())
sigma_hat_sa = np.median(np.abs(diffs_sa - np.median(diffs_sa))) / 0.67449  # MAD -> sigma
sigma2 = sigma_hat_sa ** 2

# =====================================================================
# DIAGNOSTIC CROSS-CHECK
# Compares sigma_hat from the SA series vs. the raw observed series to
# confirm seasonal adjustment actually reduced first-difference noise,
# as theoretically expected. Flags a warning if it did not.
# =====================================================================
try:
    # Load the raw observed series (pre-adjustment) for comparison. The series is in a file.
    observed = pd.read_csv(r"C:\MyPython\PM-Trends\Trends_PM10\pm10_mensual_sa\SW_IMPUTADOS_stl_sa_mensual.csv", index_col=0)
    # Choose the correct column for raw observed PM10 (not the SA column), the name of the column is "observed"
    if 'observed' not in observed.columns:
        raise KeyError("No encontré columna 'observed' en el CSV de la serie cruda.")
    # Use the 'observed' column for the raw series
    observed = observed['observed'].dropna()
        
    diffs_raw = np.diff(np.asarray(observed).ravel())
    sigma_hat_raw = np.median(np.abs(diffs_raw - np.median(diffs_raw))) / 0.67449

    reduction_pct = 100 * (1 - sigma_hat_sa / sigma_hat_raw) if sigma_hat_raw > 0 else np.nan

    print("=== sigma_hat diagnostic cross-check ===")
    print(f"sigma_hat (raw observed series) : {sigma_hat_raw:.6f}")
    print(f"sigma_hat (seasonally-adjusted) : {sigma_hat_sa:.6f}")
    print(f"Relative reduction from SA      : {reduction_pct:.1f}%")

    if sigma_hat_sa >= sigma_hat_raw:
        print("⚠️  WARNING: SA sigma_hat is NOT smaller than raw sigma_hat. "
                "Check the decomposition (possible leftover seasonality, "
                "over-differencing, or Remainder dominated by outliers).")
    elif reduction_pct < 5:
        print("⚠️  NOTE: SA reduced noise by less than 5% — seasonality "
                "signal may be weak, or Remainder still carries structure.")
    else:
        print("✅ SA series shows expected noise reduction vs. raw series.")
    print("=========================================")

except NameError:
    print("=== sigma_hat diagnostic cross-check ===")
    print(f"sigma_hat (seasonally-adjusted series): {sigma_hat_sa:.6f}")
    print("Raw 'observed' series not found in scope — skipping raw-vs-SA "
            "comparison. Define `observed` (the pre-adjustment series) to "
            "enable this cross-check.")
    print("=========================================")

# ---- BIC-style penalty bounds using the SA-based sigma_hat ----
min_penalty = 0.5 * sigma2
max_penalty = 1.0 * sigma2 * np.log(n)

print(f"\nn = {n}, sigma_hat_sa = {sigma_hat_sa:.6f}, sigma2 = {sigma2:.6f}")
print(f"Penalty search range: [{min_penalty:.4f}, {max_penalty:.4f}]")

# ---------------------------------------------------------------------------
# 2) min_size bounds for monthly series with 131-276 points (~11-23 years)
# ---------------------------------------------------------------------------
# 36 months  -> shortest plausible regime (three years) before we trust a shift
# 72 months -> six years, long enough to avoid over-fragmenting short series
min_size_lo, min_size_hi = 36, 72

# ---------------------------------------------------------------------------
# Deterministic grid (needed for a clean elbow curve) instead of random sampling
# ---------------------------------------------------------------------------
n_pen_steps = 100
n_min_size_step = 2

pen_values = np.linspace(min_penalty, max_penalty, n_pen_steps)
min_size_values = np.arange(min_size_lo, min_size_hi, n_min_size_step)

results = []
trial = 0
for min_size in min_size_values:
    for pen in pen_values:
        model = rpt.Pelt(model="clinear", min_size=int(min_size)).fit(signal)
        bkps = model.predict(pen=pen)
        n_changepoints = len(bkps) - 1

        results.append({
            'trial': trial,
            'pen': pen,
            'min_size': int(min_size),
            'n_changepoints': n_changepoints,
            'changepoint_indices': bkps[:-1],
            'changepoint_dates': [mensual.index[idx] if idx < len(mensual.index) else None
                                    for idx in bkps[:-1]]
        })
        trial += 1

results_df = pd.DataFrame(results)

# ---------------------------------------------------------------------------
# 3) Elbow method on pen vs n_changepoints (aggregated across min_size, and
#    also per min_size if you want a family of curves)
# ---------------------------------------------------------------------------
def find_elbow(x, y):
    """
    Simple, dependency-free elbow detector using the 'maximum distance to
    the chord' method between the first and last point of a monotonic curve.
    x, y must be sorted by x ascending.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Normalize to [0,1] so pen-scale and count-scale are comparable
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-12)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-12)

    p1 = np.array([x_norm[0], y_norm[0]])
    p2 = np.array([x_norm[-1], y_norm[-1]])
    line_vec = p2 - p1
    line_vec_norm = line_vec / np.linalg.norm(line_vec)

    distances = []
    for xi, yi in zip(x_norm, y_norm):
        p = np.array([xi, yi]) - p1
        proj_len = np.dot(p, line_vec_norm)
        proj_point = proj_len * line_vec_norm
        dist = np.linalg.norm(p - proj_point)
        distances.append(dist)

    elbow_idx = int(np.argmax(distances))
    return elbow_idx

# Aggregate: for each pen value, average n_changepoints across all min_size
agg = results_df.groupby('pen', as_index=False)['n_changepoints'].mean()
agg = agg.sort_values('pen').reset_index(drop=True)

elbow_idx = find_elbow(agg['pen'].values, agg['n_changepoints'].values)
elbow_pen = agg.loc[elbow_idx, 'pen']
elbow_n_changepoints = agg.loc[elbow_idx, 'n_changepoints']

print(f"Elbow point -> pen: {elbow_pen:.3f}, avg n_changepoints: {elbow_n_changepoints:.2f}")

# Plot the elbow curve
plt.figure(figsize=(9, 6))
plt.plot(agg['pen'], agg['n_changepoints'], marker='o', linestyle='-', label='Avg n_changepoints vs pen')
plt.axvline(elbow_pen, color='red', linestyle='--', label=f'Elbow pen = {elbow_pen:.2f}')
plt.xlabel('Penalty (pen)')
plt.ylabel('Average Number of Changepoints (across min_size)')
plt.title('Elbow Method: Selecting Penalty via Changepoint Count Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# Surface plot of number of changepoints for each combination of pen and min_size
# ---------------------------------------------------------------------------
pen_grid, min_size_grid = np.meshgrid(
    np.linspace(results_df['pen'].min(), results_df['pen'].max(), 100),
    np.linspace(results_df['min_size'].min(), results_df['min_size'].max(), 100)
)
n_changepoints_grid = griddata(
    (results_df['pen'], results_df['min_size']),
    results_df['n_changepoints'],
    (pen_grid, min_size_grid),
    method='cubic'
)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(pen_grid, min_size_grid, n_changepoints_grid, cmap='viridis', edgecolor='none')
#ax.set_ylabel('Pen')
ax.set_ylabel('Min Size')
#ax.set_xlabel('Min Size')
ax.set_xlabel('Pen')
ax.set_zlabel('Number of Changepoints')
ax.set_title('Surface Plot: Changepoints vs Pen and Min Size')
fig.colorbar(surface, ax=ax, label='Number of Changepoints')
plt.show()

# ---------------------------------------------------------------------------
# Save results (including elbow point) to CSV
# ---------------------------------------------------------------------------
results_df.to_csv(outfile, index=False)

elbow_summary = pd.DataFrame([{
    'elbow_pen': elbow_pen,
    'elbow_avg_n_changepoints': elbow_n_changepoints,
    'min_penalty_bound': min_penalty,
    'max_penalty_bound': max_penalty,
    'min_size_lo': min_size_lo,
    'min_size_hi': min_size_hi
}])
elbow_outfile = outfile.replace('.csv', '_elbow_summary.csv')
elbow_summary.to_csv(elbow_outfile, index=False)
print(f"Elbow summary saved to: {elbow_outfile}")

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from collections import Counter

# ---- Parameters ----
GROUPING_TOLERANCE_DAYS = 90  # breakpoints within ~3 months of each other are treated as the "same" break

# 1) Flatten all detected breakpoint dates across the n trials, keeping the trial id
all_dates = []
for _, row in results_df.iterrows():
    for d in row['changepoint_dates']:
        if d is not None:
            all_dates.append({'trial': row['trial'], 'date': d})

dates_df = pd.DataFrame(all_dates)

if not dates_df.empty:
    # 2) Convert dates to ordinal numbers so they can be clustered by elapsed time
    ordinals = dates_df['date'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)

    # 3) Single-linkage clustering: dates closer than GROUPING_TOLERANCE_DAYS merge into one cluster
    Z = linkage(ordinals, method='single', metric='euclidean')
    dates_df['cluster'] = fcluster(Z, t=GROUPING_TOLERANCE_DAYS, criterion='distance')

    # 4) Representative date per cluster: modal exact date if it repeats, otherwise the median date
    cluster_repr = {}
    for c, sub in dates_df.groupby('cluster'):
        vc = sub['date'].value_counts()
        cluster_repr[c] = vc.index[0] if vc.iloc[0] > 1 else sub['date'].sort_values().iloc[len(sub)//2]
    dates_df['cluster_date'] = dates_df['cluster'].map(cluster_repr)

    # 5) Frequency with which each grouped breakpoint appeared across the n trials
    n_trials = len(results_df)  # total number of (pen, min_size) combinations actually evaluated
                                
    cluster_freq = (dates_df.groupby('cluster')
                        .agg(cluster_date=('cluster_date', 'first'),
                                n_trials = ('trial', 'nunique')  # total number of (pen, min_size) combinations actually evaluated
                            )
                            .assign(pct_trials=lambda d: 100 * d['n_trials'] / n_trials)
                            .sort_values('n_trials', ascending=False)
                            .reset_index(drop=True))

    # 6) Per-trial "signature" = sorted tuple of cluster labels -> identifies identical segmentations
    trial_signature = dates_df.groupby('trial')['cluster'].apply(lambda x: tuple(sorted(x)))
    signature_counts = Counter(trial_signature)

    # 7) Modal segmentation = most frequent signature, and its occurrence rate
    modal_signature, modal_count = signature_counts.most_common(1)[0]
    modal_pct = 100 * modal_count / n_trials
    modal_dates = sorted(cluster_repr[c] for c in modal_signature)

    print(f"Modal segmentation occurred in {modal_count}/{n_trials} trials ({modal_pct:.1f}%).")
    print("Breakpoint dates of the modal segmentation:")
    for d in modal_dates:
        print(" -", d.strftime('%Y-%m'))

    # 8) Save for additional analysis and reporting
    dates_df.to_csv(outfile.replace('.csv', '_breakpoint_grouping.csv'), index=False)
    cluster_freq.to_csv(outfile.replace('.csv', '_breakpoint_frequency.csv'), index=False)
else:
    print("No changepoints were detected in any of the trials.")